# Hedging European Gas Futures — a hands-on tutorial

> *"The market can stay irrational longer than you can stay solvent."* — attributed to J. M. Keynes,
> and quoted by every gas trader who lived through 2022.

## The situation you are in

Your firm makes markets in gas across many venues. Every fill leaves a little risk behind: a few lots
long Dec-26 here, a few short Q1-27 there. Nobody *chose* those positions — they are the residue of
providing liquidity. The desk has hedging flows for power, oil and coal, but **nothing for European gas
on ICE**. That is the gap you have been asked to fill: take the gas risk that accumulates elsewhere and
neutralise it, cheaply and reliably, on the exchange.

This tutorial equips you to do that. It is built for someone who is a strong engineer, knows what a
standard deviation is, has run e-trading risk controls, but has never had to *decide* what to hedge with,
how much, and when.

## A note on the venue and the hubs (read this first)

* The European benchmark gas future is the **Dutch TTF future traded on ICE Endex** (ICE's continental
  European exchange, legally seated in Amsterdam; ICE root code `TFM`). It is the most liquid gas
  contract in Europe by a wide margin and the instrument you will most often hedge *with*.
* The **French hub is PEG** (Point d'Échange de Gaz). PEG futures have historically been listed on the
  Paris-born venue Powernext (now part of EEX); ICE also lists PEG products. In this tutorial "the risk we
  inherit" is frequently PEG-denominated, and "the instrument we hedge with" is frequently TTF — because
  that mismatch is exactly what makes hedging interesting and is a daily reality for continental desks.
* **Verify product codes, lot sizes and expiry rules against the current ICE product specification**
  before writing production code. Everything mechanical in this tutorial (month codes, delivery hours,
  hours-weighted strips) is standard and checkable; everything about liquidity levels is a *stylised
  estimate* and is clearly marked as such.

## One correction before we begin

You mentioned that gas futures are "listed for even months". That is a pattern from some *oil* and
*metals* contracts. **European gas futures are listed for every calendar month**, typically several years
out, *plus* tradable strips (quarters, seasons, calendar years) that settle into those months. What *is*
true is that liquidity is very uneven along the curve — which is where the fun starts (Chapter 1.3).

## How the tutorial is organised

Three modules, each a chain of chapters. Every chapter opens with the **problem** it solves, builds the
**solution** from first principles with code and plots, and closes with the **pitfall** that motivates
the next chapter. Chapters end with non-trivial questions whose answers are hidden until you click.

| Module | Question it answers | Chapters |
|---|---|---|
| **1. Market structure & liquidity** | *What exactly am I trading, and where is it liquid?* | [1.1 Anatomy of a gas future](module1_market_structure/01_anatomy_of_a_gas_future.ipynb) · [1.2 The curve and its strips](module1_market_structure/02_curve_and_strips.ipynb) · [1.3 Where the liquidity lives](module1_market_structure/03_liquidity_profile.ipynb) · [1.4 How the curve moves](module1_market_structure/04_curve_dynamics.ipynb) · [1.5 Hubs and basis](module1_market_structure/05_hubs_and_basis.ipynb) |
| **2. Hedging core** | *What should I hedge with, how, and when?* | [2.1 Why hedge, and what "good" means](module2_hedging_core/01_why_hedge.ipynb) · [2.2 The instrument menu](module2_hedging_core/02_instrument_menu.ipynb) · [2.3 The 1:1 hedge and its three mismatches](module2_hedging_core/03_one_to_one_and_mismatches.ipynb) · [2.4 Stack-and-roll vs strip hedges](module2_hedging_core/04_stack_and_roll.ipynb) · [2.5 When to hedge: bands, costs, frequency](module2_hedging_core/05_hedge_timing_and_cost.ipynb) · [2.6 Proxy hedging and the hedging pipeline](module2_hedging_core/06_proxy_hedging_pipeline.ipynb) |
| **3. Risk analysis methods** | *How much, exactly — and how wrong could I be?* | [3.1 The linear model](module3_risk_analysis/01_linear_model.ipynb) · [3.2 The minimum-variance hedge ratio](module3_risk_analysis/02_minimum_variance_hedge.ipynb) · [3.3 Many instruments and PCA](module3_risk_analysis/03_multi_instrument_and_pca.ipynb) · [3.4 Measuring what is left](module3_risk_analysis/04_residual_risk_var.ipynb) · [3.5 Living with estimation error](module3_risk_analysis/05_estimation_error_and_regimes.ipynb) |
| **Appendices** | The maths that would break the flow | [A. Least squares, derived](appendix/A_least_squares.ipynb) · [B. Delivery hours, DST and strip arithmetic](appendix/B_delivery_hours_and_strips.ipynb) · [C. Notation, PCA and references](appendix/C_notation_pca_references.ipynb) |

The "linear model" someone mentioned to you is the spine of Module 3 — and it is *exactly* right for a
futures book, as Chapter 3.1 shows.

## How to run it

* Open any notebook in this folder with the project's `.venv` kernel and run top to bottom.
* All shared code lives in the `gashedge` package (`contracts`, `market_data`, `hedging`, `risk`,
  `plotting`). Notebooks call it; they do not re-implement it. Read the source — it is short and documented.
* Logging is via the `logging` module with millisecond timestamps; there are no `print` statements.
* Every random quantity is seeded. Change the seed and the story should still hold; if it doesn't, that is
  itself a lesson (Chapter 3.5).

## Conventions

| Symbol | Meaning |
|---|---|
| $S$ | price of the exposure we hold (EUR/MWh) |
| $F$ | price of the instrument we hedge with (EUR/MWh) |
| $Q_S$ | exposure size in MWh, positive = long gas |
| $h$ | hedge ratio: MWh of $F$ sold per MWh of $S$ held |
| $\Delta X$ | one-day change $X_{t+1}-X_t$ |
| $\Pi$ | portfolio P&L in EUR |
| $\sigma, \rho, \Sigma$ | standard deviation, correlation, covariance matrix |

The full notation table and references live in [Appendix C](appendix/C_notation_pca_references.ipynb).

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "gashedge").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from datetime import date
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gashedge import get_logger
from gashedge.plotting import setup_style
from gashedge.contracts import contract_table
setup_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
log = get_logger("intro")
log.info("Environment ready")

2026-09-13 16:46:47.198 | INFO    | intro | Environment ready


## A five-line preview

Below is the smallest possible demonstration of the machinery: the front three TTF contracts listed
today, with their symbols, delivery hours and expiries. If this runs, your environment is ready.

In [2]:
AS_OF = date(2026, 9, 14)
table = contract_table(AS_OF, n_months=3)
log.info("Front three contracts as of %s: %s", AS_OF, ", ".join(table.symbol))
table[["tenor", "symbol", "delivery_start", "hours", "lot_mwh", "expiry", "days_to_expiry"]]

2026-09-13 16:46:47.202 | INFO    | intro | Front three contracts as of 2026-09-14: TFMV26, TFMX26, TFMZ26


,tenor,symbol,delivery_start,hours,lot_mwh,expiry,days_to_expiry
0,1,TFMV26,2026-10-01,745,745.0,2026-09-29,15
1,2,TFMX26,2026-11-01,720,720.0,2026-10-28,44
2,3,TFMZ26,2026-12-01,744,744.0,2026-11-27,74


Now go to [Chapter 1.1](module1_market_structure/01_anatomy_of_a_gas_future.ipynb). Take your time; the
chapters are designed to be read in order, and each one earns the next.